# Live single-file explorer

Interactive inspection of **one EDF file** at a time — load it once, then explore it live.

**Workflow:**
1. **Section 1 — Load**: pick an EDF (required), a hypnogram `.txt` (optional), and optionally a `remap_reref_persubject.json` config (from *select&remap_channels_edf*). Choose the EEG analysis channels and the EOG/EMG scoring channels, then load. A **raw / preprocessed** toggle governs every plot.
2. **Section 2 — Overview**: whole-recording quality plots per EEG electrode (histogram, time series, metrics, hypnospectrogram, whole-night PSD, per-stage PSD).
3. **Section 3 — Epoch explorer**: navigate 30 s epochs with the scoring montage (EEG + EOG + EMG), event annotations, per-epoch PSD, and optional **rescoring** (saved to a new `_rescored.txt`, never overwriting the original).
4. **Section 4 — Quick rejection**: compute the 5 rejection methods live, view the heatmap, browse rejected epochs, and compare per-stage clean vs. rejected PSD.

> This is a quality control + scoring-review companion. It does not modify the EDF and writes outputs only when you click an explicit **Save** button.

In [ ]:
import os
import re
import io
import json
import base64
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as mpe

import mne
import yasa
import chardet
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser
from scipy.stats import kurtosis as scipy_kurtosis, skew as scipy_skew
from scipy.signal import find_peaks, savgol_filter, spectrogram as scipy_spectrogram

try:
    from specparam import SpectralModel
    _HAS_SPECPARAM = True
except Exception:
    _HAS_SPECPARAM = False

# ipyevents enables keyboard navigation in Section 3; optional (buttons still work without it).
try:
    from ipyevents import Event as _IpyEvent
    _HAS_IPYEVENTS = True
except Exception:
    _HAS_IPYEVENTS = False

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# YASA integer codes for AASM stages (used for the hypnospectrogram and per-stage grouping).
STAGE_MAP = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
STAGES = ['W', 'N1', 'N2', 'N3', 'R']
# Frequency bands for PSD annotation (non-overlapping, covers 0.5-30 Hz).
BANDS = [
    (0.5,  4,  'δ',  '#cce0f5'),
    (4,    8,  'θ',  '#ccf0d8'),
    (8,   12,  'α',  '#fff2cc'),
    (12,  16,  'σ',  '#f5d0f0'),
    (16,  30,  'β',  '#ffbaba'),
]

In [ ]:
# ----- EDF header parsing (custom binary parser, header only — never loads signal data) -----
# Same parser as inspect_edf / select&remap_channels_edf. pyedflib was too strict for the
# heterogeneous EDF files in real datasets (encoding edge cases, non-standard headers).

def detect_encoding(byte_string, min_confidence=0.6):
    result = chardet.detect(byte_string)
    encoding = result['encoding']
    confidence = result['confidence']
    if encoding is None or confidence < min_confidence:
        # latin-1 maps every byte, so it never raises — a safe fallback for odd headers.
        return 'latin-1'
    return encoding


def read_edf_header_custom(file_path):
    """Read EDF header fields octet by octet (no signal data). Returns a dict of lists per channel."""
    with open(file_path, 'rb') as f:
        header = {}
        raw_header = f.read(256)
        encoding = detect_encoding(raw_header)
        f.seek(0)
        header['version'] = f.read(8).decode(encoding).strip()
        header['patient_id'] = f.read(80).decode(encoding).strip()
        header['recording_id'] = f.read(80).decode(encoding).strip()
        header['start_date'] = f.read(8).decode(encoding).strip()
        header['start_time'] = f.read(8).decode(encoding).strip()
        header['header_bytes'] = int(f.read(8).decode(encoding).strip())
        header['reserved'] = f.read(44).decode(encoding).strip()
        header['n_data_records'] = int(f.read(8).decode(encoding).strip())
        header['duration_data_record'] = float(f.read(8).decode(encoding).strip())
        header['n_channels'] = int(f.read(4).decode(encoding).strip())
        n = header['n_channels']
        lengths = {
            'channel': 16, 'transducer_type': 80, 'dimension': 8,
            'physical_min': 8, 'physical_max': 8, 'digital_min': 8, 'digital_max': 8,
            'prefiltering': 80, 'samples_per_record': 8, 'reserved': 32,
        }
        channel_fields = {k: [] for k in lengths}
        for key, length in lengths.items():
            channel_fields[key] = [f.read(length).decode(encoding).strip() for _ in range(n)]
        header.update(channel_fields)
        # The per-channel 8-byte field is the NUMBER OF SAMPLES PER DATA RECORD, not the rate.
        # sampling_frequency = samples_per_record / duration_data_record (e.g. 40/0.1 = 400 Hz).
        dur = header['duration_data_record']
        sf_list = []
        for s in header['samples_per_record']:
            try:
                sf = float(s) / dur
                sf_list.append(str(int(sf)) if sf == int(sf) else str(sf))
            except (ValueError, ZeroDivisionError):
                sf_list.append(s)
        header['sampling_frequency'] = sf_list
    return header


def load_json_lenient(path):
    """Parse JSON strictly; on failure, repair a single trailing comma before }/] and retry."""
    text = Path(path).read_text(encoding='utf-8')
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        repaired = re.sub(r',(\s*[}\]])', r'\1', text)
        return json.loads(repaired)


# ----- Channel-type detection (transducer-type + channel-name; from inspect_edf_voila_EMG) -----
# Anchored full 10-10 system + mastoids + literal 'EEG' (matches select&remap_channels_edf).
KNOWN_EEG_CHANNEL_RE = (
    r'\bFp1\b|\bFpz\b|\bFp2\b|\bAF7\b|\bAF3\b|\bAFz\b|\bAF4\b|\bAF8\b|\bF7\b|\bF5\b|\bF3\b|\bF1\b'
    r'|\bFz\b|\bF2\b|\bF4\b|\bF6\b|\bF8\b|\bFT7\b|\bFC5\b|\bFC3\b|\bFC1\b|\bFCz\b|\bFC2\b|\bFC4\b'
    r'|\bFC6\b|\bFT8\b|\bT7\b|\bC5\b|\bC3\b|\bC1\b|\bCz\b|\bC2\b|\bC4\b|\bC6\b|\bT8\b|\bTP7\b|\bCP5\b'
    r'|\bCP3\b|\bCP1\b|\bCPz\b|\bCP2\b|\bCP4\b|\bCP6\b|\bTP8\b|\bP7\b|\bP5\b|\bP3\b|\bP1\b|\bPz\b'
    r'|\bP2\b|\bP4\b|\bP6\b|\bP8\b|\bPO7\b|\bPO5\b|\bPO3\b|\bPOz\b|\bPO4\b|\bPO6\b|\bPO8\b|\bO1\b'
    r'|\bOz\b|\bO2\b|\bM1\b|\bM2\b|EEG'
)


def detect_channel_types(channels, transducers):
    """Classify each channel as EEG / EOG / EMG / ECG / other from name + transducer type.
    EOG: transducer~EOG or name~eog. EMG: transducer~EMG or name~emg|chin|menton.
    ECG: transducer~ECG or name~ecg. EEG: known 10-10 name or transducer EEG/AGAGCL,
    excluding anything already matched as eog/ecg/emg. Returns dict name -> type."""
    types = {}
    for ch, td in zip(channels, transducers):
        td = td or ''
        is_eog = bool(re.search(r'EOG', td, re.I)) or bool(re.search(r'eog', ch, re.I))
        is_emg = bool(re.search(r'EMG', td, re.I)) or bool(re.search(r'emg|chin|menton', ch, re.I))
        is_ecg = bool(re.search(r'ECG|EKG', td, re.I)) or bool(re.search(r'ecg|ekg', ch, re.I))
        is_eeg = (bool(re.search(r'\bEEG\b|\bAGAGCL ELECTRODE\b', td, re.I))
                  or bool(re.search(KNOWN_EEG_CHANNEL_RE, ch, re.I)))
        if is_eog:
            types[ch] = 'EOG'
        elif is_emg:
            types[ch] = 'EMG'
        elif is_ecg:
            types[ch] = 'ECG'
        elif is_eeg:
            types[ch] = 'EEG'
        else:
            types[ch] = 'other'
    return types

In [ ]:
# ----- Channel handling + quality metrics (copied from quality_overview_voila, kept identical) -----

def odd(x):
    """Return nearest odd integer to x (required by savgol_filter)."""
    n = int(round(x))
    if n % 2 != 0:
        return n
    return n - 1 if abs(x - (n - 1)) <= abs((n + 1) - x) else n + 1


def drop_suffix_duplicates(raw):
    """Keep only the -0 variant when MNE creates -0/-1 duplicates for repeated channel names."""
    groups = {}
    for ch in raw.ch_names:
        if ch.endswith('-0') or ch.endswith('-1'):
            base = ch.rsplit('-', 1)[0]
            groups.setdefault(base, []).append(ch)
    to_drop = []
    for base, ch_list in groups.items():
        if any(c.endswith('-0') for c in ch_list):
            to_drop.extend(c for c in ch_list if not c.endswith('-0'))
    if to_drop:
        raw.drop_channels(to_drop)
    return raw, to_drop


def adapt_remap_dict_to_suffixes(raw, remap_dict):
    """Handle MNE's -0 suffix when the remap config uses the base channel name."""
    ch_set = set(raw.ch_names)
    new_remap = {}
    for base_label, target in remap_dict.items():
        if base_label in ch_set:
            new_remap[base_label] = target
        elif f'{base_label}-0' in ch_set:
            new_remap[f'{base_label}-0'] = target
    return new_remap


def get_phys_bounds_uV(extras, ch_idx):
    """Reconstruct physical_min/max (µV) from MNE _raw_extras (MNE 1.9 stores physical_max + offset)."""
    phys_max = float(extras['physical_max'][ch_idx])
    offset = float(extras['offsets'][ch_idx])
    return -phys_max + 2 * offset, phys_max


def compute_signal_metrics(sig_uV, phys_min_uV, phys_max_uV):
    """Compute all quality metrics for one channel (unfiltered signal, in µV)."""
    cur_std = float(np.std(sig_uV))
    u = np.unique(sig_uV)
    # Flat signal: tolerance = 2x ADC resolution to handle rounding, minimum 0.06 µV.
    if len(u) > 1:
        res = float(np.min(np.abs(np.diff(u))))
        flat_atol = max(2 * res, 0.06)
    else:
        flat_atol = 0.06
    flat_pct = float(np.isclose(np.diff(sig_uV), 0, atol=flat_atol, rtol=0.0).mean()) * 100
    # EDF physical bounds: fraction of samples at/near the header declared limits (hard clipping).
    if not (np.isnan(phys_min_uV) or np.isnan(phys_max_uV)):
        bounds_pct = float(((sig_uV <= phys_min_uV + 0.5) | (sig_uV >= phys_max_uV - 0.5)).mean()) * 100
    else:
        bounds_pct = 0.0
    # Histogram + Savitzky-Golay smoothing + peak detection.
    if cur_std < 1e-10 or len(u) < 3:
        histo, edges = np.histogram(sig_uV, bins=10)
        sg_window = 5
    else:
        histo, edges = np.histogram(sig_uV, bins='scott')
        sg_window = max(5, odd(0.02 * len(histo)))
    bin_centers = (edges[:-1] + edges[1:]) / 2
    y_smooth = savgol_filter(histo.astype(float), window_length=sg_window, polyorder=3)
    prom_thresh = 0.25 * float(np.max(y_smooth)) if np.max(y_smooth) > 0 else 0.0
    peaks, _ = find_peaks(y_smooth, prominence=prom_thresh)
    total = float(np.sum(histo))
    hist_extreme_pct = float((histo[0] + histo[-1]) / total * 100) if total > 0 else 0.0
    abs_uV = np.abs(sig_uV)
    return {
        'mean_uV': float(np.mean(sig_uV)),
        'std_uV': cur_std,
        'kurtosis': float(scipy_kurtosis(sig_uV, fisher=True)),
        'skewness': float(scipy_skew(sig_uV)),
        'p99_abs_uV': float(np.percentile(abs_uV, 99.0)),
        'p999_abs_uV': float(np.percentile(abs_uV, 99.9)),
        'flat_pct': flat_pct,
        'bounds_pct': bounds_pct,
        'hist_extreme_pct': hist_extreme_pct,
        'n_peaks': int(len(peaks)),
        'histo': histo, 'edges': edges, 'bin_centers': bin_centers,
        'y_smooth': y_smooth, 'peaks': peaks,
    }


def flag_channel(metrics, thresholds):
    """Return list of flag reasons for a channel, or empty list if within all thresholds."""
    reasons = []
    if metrics['flat_pct'] > thresholds['flat_pct']:
        reasons.append(f"flat_pct={metrics['flat_pct']:.1f}% > {thresholds['flat_pct']}%")
    if metrics['bounds_pct'] > thresholds['bounds_pct']:
        reasons.append(f"bounds_pct={metrics['bounds_pct']:.2f}% > {thresholds['bounds_pct']}%")
    if metrics['n_peaks'] >= thresholds['n_peaks']:
        shape = 'bimodal' if metrics['n_peaks'] == 2 else 'multimodal'
        reasons.append(f"n_peaks={metrics['n_peaks']} ({shape})")
    if metrics['std_uV'] < thresholds['std_low']:
        reasons.append(f"std={metrics['std_uV']:.2f} µV < {thresholds['std_low']} µV (dead/flat channel)")
    if metrics['hist_extreme_pct'] > thresholds['hist_extreme_pct']:
        reasons.append(f"hist_extreme={metrics['hist_extreme_pct']:.2f}% > {thresholds['hist_extreme_pct']}%")
    return reasons

In [ ]:
# ----- Plot helpers (histogram / time series / PSD from quality_overview, + new PSD variants) -----

def plot_histogram_figure(metrics, ch_name, y_max=None, x_lim=None):
    """Signal amplitude distribution with Savitzky-Golay smooth + detected peaks."""
    edges, histo = metrics['edges'], metrics['histo']
    bin_centers, y_smooth, peaks = metrics['bin_centers'], metrics['y_smooth'], metrics['peaks']
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(edges[:-1], histo, width=edges[1] - edges[0], align='edge',
           color='#d4e8f5', edgecolor='#7ab4d0')
    ax.plot(bin_centers, y_smooth, color='#6a51a3', linewidth=2, label='Savitzky-Golay smooth')
    ax.plot(bin_centers[peaks], y_smooth[peaks], 'r+', markersize=10, markeredgewidth=2,
            label=f'{len(peaks)} peak(s) detected')
    ax.set_xlabel('Amplitude (μV)', fontsize=11)
    ax.set_ylabel('Counts', fontsize=11)
    if y_max is not None:
        ax.set_ylim(0, y_max * 1.05)
    if x_lim is not None:
        ax.set_xlim(-x_lim, x_lim)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    ax.set_title(f'{ch_name} — amplitude histogram', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    return fig


def _annotate_bands(ax, fmax):
    """Shade the EEG frequency bands + mark 50 Hz on a PSD axis."""
    for flo, fhi, label, color in BANDS:
        fhi_c = min(fhi, fmax)
        if flo >= fmax:
            continue
        ax.axvspan(flo, fhi_c, alpha=0.20, color=color, zorder=0, linewidth=0)
        ax.text((flo + fhi_c) / 2, 0.97, label, ha='center', va='top', fontsize=8,
                color='#555', transform=ax.get_xaxis_transform())
    if fmax >= 50:
        ax.axvline(50, color='#c0392b', linewidth=0.8, linestyle='--', alpha=0.6)


def welch_psd(sig_uV, sf, fmin=0.5, fmax=None):
    """Welch PSD (4 s Hamming windows, 50% overlap) on a 1D signal in µV. Returns (freqs, psd µV²/Hz)."""
    if fmax is None:
        fmax = min(100.0, sf / 2 - 0.5)
    n_per_seg = int(4 * sf)
    n_overlap = n_per_seg // 2
    psds, freqs = mne.time_frequency.psd_array_welch(
        sig_uV[np.newaxis, :], sfreq=sf, fmin=fmin, fmax=fmax,
        n_per_seg=n_per_seg, n_overlap=n_overlap, n_fft=n_per_seg, verbose=False)
    return freqs, psds[0]


def plot_psd_figure(sig_uV, sf, title='PSD'):
    """Whole-recording PSD on a dB axis with band shading."""
    freqs, psd = welch_psd(sig_uV, sf)
    psd_db = 10 * np.log10(np.maximum(psd, 1e-10))
    fig, ax = plt.subplots(figsize=(7, 3))
    _annotate_bands(ax, float(freqs[-1]))
    ax.plot(freqs, psd_db, linewidth=1.0, color='#2a6099', zorder=2)
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB/Hz)', fontsize=11)
    ax.set_xlim(0.5, float(freqs[-1]))
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    return fig


def plot_timeseries_figure(sig_uV, sf, y_lim, title='time series'):
    """Whole-recording time series (downsampled to ~10 Hz for display), fixed Y-axis."""
    step = max(1, int(sf / 10))
    t_h = np.arange(0, len(sig_uV), step) / sf / 3600
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.plot(t_h, sig_uV[::step], linewidth=0.3, color='#2a6099', rasterized=True)
    ax.set_xlim(0, len(sig_uV) / sf / 3600)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel('Time (h)', fontsize=11)
    ax.set_ylabel('µV', fontsize=11)
    ax.axhline(0, color='#bbb', linewidth=0.5, linestyle='--')
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    return fig


def compute_epoch_psds(sig_uV, sf, n_epochs, fmin=0.5, fmax=30.0):
    """Per-epoch Welch PSD on a 1D signal. Returns (freqs, psds[n_epochs, n_freqs]) in µV²/Hz.
    Each 30 s epoch is one Welch window set; epochs beyond n_epochs*30 s are ignored."""
    samp_per_epoch = int(round(30 * sf))
    usable = n_epochs * samp_per_epoch
    seg = sig_uV[:usable].reshape(n_epochs, samp_per_epoch)
    fmax = min(fmax, sf / 2 - 0.5)
    n_per_seg = int(4 * sf)
    n_overlap = n_per_seg // 2
    psds, freqs = mne.time_frequency.psd_array_welch(
        seg, sfreq=sf, fmin=fmin, fmax=fmax,
        n_per_seg=n_per_seg, n_overlap=n_overlap, n_fft=n_per_seg, verbose=False)
    return freqs, psds


def plot_perstage_psd_figure(sig_uV, sf, hypno_labels, ch_name, mask_keep=None):
    """Mean PSD per sleep stage (one curve per W/N1/N2/N3/R) for one channel.
    hypno_labels: array of stage strings per 30 s epoch. mask_keep: optional bool array
    (per epoch) to restrict which epochs are averaged (used for clean-vs-rejected views)."""
    n_epochs = len(hypno_labels)
    freqs, psds = compute_epoch_psds(sig_uV, sf, n_epochs)
    psds_db = 10 * np.log10(np.maximum(psds, 1e-10))
    colors = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba', 'N3': '#6a51a3', 'R': '#c994c7'}
    fig, ax = plt.subplots(figsize=(7, 3))
    _annotate_bands(ax, float(freqs[-1]))
    for st in STAGES:
        sel = (np.asarray(hypno_labels) == st)
        if mask_keep is not None:
            sel = sel & mask_keep
        if sel.sum() == 0:
            continue
        ax.plot(freqs, psds_db[sel].mean(axis=0), linewidth=1.4, color=colors[st],
                label=f'{st} (n={int(sel.sum())})')
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB/Hz)', fontsize=11)
    ax.set_xlim(0.5, float(freqs[-1]))
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    ax.set_title(f'{ch_name} — mean PSD per stage', fontsize=11)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    return fig


def make_spectrogram_figure(sig_uV, sf, hypno_vec):
    """YASA hypnospectrogram on a 0.1-40 Hz bandpassed copy (display filter only)."""
    sig_filt = mne.filter.filter_data(
        sig_uV[np.newaxis, :], sfreq=sf, l_freq=0.1, h_freq=40,
        method='fir', phase='zero-double', fir_window='hamming', verbose=False)[0]
    hypno_full = yasa.hypno_upsample_to_data(hypno=hypno_vec, sf_hypno=1 / 30, data=sig_filt, sf_data=sf)
    fig = yasa.plot_spectrogram(sig_filt, sf, hypno_full, fmin=0.5, fmax=40)
    fig.set_size_inches(10, 3.5)
    for ax in fig.axes:
        ax.tick_params(labelsize=8)
        ax.xaxis.label.set_fontsize(11)
        ax.yaxis.label.set_fontsize(11)
    fig.tight_layout()
    return fig

In [ ]:
# ----- Epoch rejection (5 methods + heatmap + summary; from preprocessing_voila) -----

def compute_rejection_masks(epochs_data_uV, sfreq, hypno_epochs, ptp_thresholds,
                            flat_thresh, grad_thresh,
                            psd_freqs, psds_data_uV2, thresh_error, thresh_r2):
    """Per-(epoch, channel) boolean rejection masks for each method.
    epochs_data_uV: (n_epochs, n_channels, n_times) in µV. Returns dict of bool arrays
    keyed 'amplitude','flat','gradient','1f_error','1f_r2'."""
    n_epochs, n_channels, _ = epochs_data_uV.shape
    ptp = np.ptp(epochs_data_uV, axis=-1)  # (n_epochs, n_channels)
    ptp_thresh_arr = np.array([ptp_thresholds.get(s, 250.0) for s in hypno_epochs])[:, np.newaxis]
    mask_amplitude = ptp > ptp_thresh_arr
    mask_flat = ptp < flat_thresh
    grad = np.max(np.abs(np.diff(epochs_data_uV, axis=-1)), axis=-1)
    mask_gradient = grad > grad_thresh

    mask_1f_error = np.zeros((n_epochs, n_channels), dtype=bool)
    mask_1f_r2 = np.zeros((n_epochs, n_channels), dtype=bool)
    # 1/f fit restricted to >=2 Hz; a full peak model removes spindle/alpha power before
    # evaluating the aperiodic fit quality (max_n_peaks=0 would over-reject N2/REM).
    if _HAS_SPECPARAM and psds_data_uV2 is not None and psd_freqs is not None:
        freq_mask = psd_freqs >= 2
        freqs_reduced = psd_freqs[freq_mask]
        sp_model = SpectralModel(peak_width_limits=[0.5, 20], aperiodic_mode='fixed', min_peak_height=0.3)
        for ei in range(n_epochs):
            for ci in range(n_channels):
                try:
                    sp_model.fit(freqs_reduced, psds_data_uV2[ei, ci, freq_mask])
                    error = sp_model.get_metrics('error', 'mae')
                    r2 = sp_model.get_metrics('gof', 'squared')
                    if error > thresh_error:
                        mask_1f_error[ei, ci] = True
                    if r2 < thresh_r2:
                        mask_1f_r2[ei, ci] = True
                except Exception:
                    mask_1f_error[ei, ci] = True
                    mask_1f_r2[ei, ci] = True
    return {'amplitude': mask_amplitude, 'flat': mask_flat, 'gradient': mask_gradient,
            '1f_error': mask_1f_error, '1f_r2': mask_1f_r2}


REJECT_METHODS = ['amplitude', 'flat', 'gradient', '1f_error', '1f_r2']


def build_combined_method_matrix(mask_dict):
    """(n_epochs, n_channels) int matrix: 0=none 1=amp 2=flat 3=grad 4=1f_err 5=1f_r2 6=multiple."""
    code_map = {name: i + 1 for i, name in enumerate(REJECT_METHODS)}
    shape = next(iter(mask_dict.values())).shape
    combined = np.zeros(shape, dtype=int)
    for name, code in code_map.items():
        combined[mask_dict[name]] = code
    n_flags = sum(mask_dict[name].astype(int) for name in REJECT_METHODS)
    combined[n_flags > 1] = 6
    return combined


def plot_rejection_heatmap(combined_matrix, ch_names, hypno_epochs, file_id):
    """Channels x epochs heatmap coloured by rejection method, with a hypnogram strip on top."""
    COLORS = ['#1c0a3b', '#c0392b', '#2980b9', '#e67e22', '#f1c40f', '#27ae60', '#7b0000']
    LABELS = ['none', 'amplitude', 'flat', 'gradient', '1/f error', '1/f R2', 'multiple']
    cmap = mcolors.ListedColormap(COLORS)
    norm = mcolors.BoundaryNorm(np.arange(-0.5, 7.5, 1), cmap.N)
    STAGE_H = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    hypno_y = np.array([STAGE_H.get(s, -0.5) for s in hypno_epochs])
    n_channels = len(ch_names)
    n_epochs = combined_matrix.shape[0]
    pct_rejected = 100.0 * combined_matrix.any(axis=1).mean()
    fig_w = max(8, min(n_epochs / 6, 20))
    fig_h = max(3, 0.45 * n_channels + 2.0)
    fig, axes = plt.subplots(2, 1, figsize=(fig_w, fig_h),
                             gridspec_kw={'height_ratios': [1.2, n_channels], 'hspace': 0.04})
    ax_hyp = axes[0]
    x_step = np.arange(n_epochs + 1)
    y_step = np.append(hypno_y, hypno_y[-1])
    ax_hyp.step(x_step, y_step, where='post', color='#555555', linewidth=1.2)
    for ei in range(n_epochs):
        if hypno_epochs[ei] == 'R':
            ax_hyp.plot([ei, ei + 1], [STAGE_H['R'], STAGE_H['R']], color='#c0392b',
                        linewidth=2.5, solid_capstyle='butt')
    ax_hyp.set_yticks([0, 1, 2, 3, 4])
    ax_hyp.set_yticklabels(['N3', 'N2', 'N1', 'REM', 'W'], fontsize=7)
    ax_hyp.set_ylim(-0.8, 4.8)
    ax_hyp.set_xlim(0, n_epochs)
    ax_hyp.set_xticks([])
    for sp in ['top', 'right', 'bottom']:
        ax_hyp.spines[sp].set_visible(False)
    ax = axes[1]
    im = ax.pcolormesh(np.arange(n_epochs + 1), np.arange(n_channels + 1),
                       combined_matrix.T, cmap=cmap, norm=norm, rasterized=True)
    ax.set_yticks(np.arange(n_channels) + 0.5)
    ax.set_yticklabels(ch_names, fontsize=8)
    ax.set_xlabel('Epoch index', fontsize=9)
    ax.set_xlim(0, n_epochs)
    ax.set_ylim(0, n_channels)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.02, pad=0.01, ticks=np.arange(7))
    cbar.ax.set_yticklabels(LABELS, fontsize=7)
    fig.suptitle(f'{file_id} — Artefacts ({pct_rejected:.0f}% rejected)', fontsize=11, y=1.01)
    fig.tight_layout()
    return fig


def build_rejection_summary(mask_dict, hypno_epochs, file_id):
    """Per-stage, per-method rejection counts DataFrame."""
    rows = []
    for stage in STAGES:
        stage_mask = (np.asarray(hypno_epochs) == stage)
        n_total = int(stage_mask.sum())
        any_flag = np.zeros(len(hypno_epochs), dtype=bool)
        for name in REJECT_METHODS:
            any_flag |= mask_dict[name].any(axis=1)
        n_rej_any = int((any_flag & stage_mask).sum())
        rows.append({'file_id': file_id, 'stage': stage, 'method': 'any',
                     'n_total': n_total, 'n_rejected': n_rej_any,
                     'pct_rejected': round(100 * n_rej_any / n_total, 1) if n_total > 0 else float('nan')})
        for name in REJECT_METHODS:
            per_epoch = mask_dict[name].any(axis=1)
            n_rej = int((per_epoch & stage_mask).sum())
            rows.append({'file_id': file_id, 'stage': stage, 'method': name,
                         'n_total': n_total, 'n_rejected': n_rej,
                         'pct_rejected': round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan')})
    return pd.DataFrame(rows)

In [ ]:
# ----- Shared state + small helpers used across all sections -----

# Single mutable container shared by every section. Populated by Section 1 "Load".
STATE = {
    'edf_path': None, 'hypno_path': None,
    'raw_raw': None, 'raw_proc': None,         # cached Raw objects (raw vs preprocessed)
    'eeg_channels': [], 'eog_channels': [], 'emg_channels': [],
    'sf': None, 'n_epochs': 0,
    'hypno_labels': None,                       # np.array of stage strings (one per 30 s epoch) or None
    'rescored_labels': None,                    # working copy edited in Section 3
    'events': None,                             # DataFrame Name/Start/Duration (seconds) or None
    'phys_bounds': {},                          # remapped EEG name -> (phys_min_uV, phys_max_uV)
    'final_to_orig': {},                        # remapped EEG name -> original base name
    'reject_masks': None,                       # dict of method masks (Section 4)
    'cur_epoch': 0,
}


def make_signal_toggle():
    """A Raw/Preprocessed selector. Each section owns one so the source is explicit per section."""
    return widgets.ToggleButtons(options=['Raw', 'Preprocessed'], value='Raw',
                                 description='Signal:', style={'description_width': 'initial'})


def raw_for(source):
    """Return the Raw object for a given signal source ('Raw' or 'Preprocessed')."""
    if source == 'Preprocessed' and STATE['raw_proc'] is not None:
        return STATE['raw_proc']
    return STATE['raw_raw']


def fig_to_png_bytes(fig):
    """Render a matplotlib figure to PNG bytes and close it (static-PNG rendering)."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf.read()


def show_fig(fig, out):
    """Display a figure as a PNG image inside an Output widget (replaces previous content)."""
    png = fig_to_png_bytes(fig)
    with out:
        clear_output(wait=True)
        display(widgets.Image(value=png, format='png'))


def labels_to_yasa_vec(labels):
    """Map an array of AASM stage strings to YASA integer codes (unknown -> NaN)."""
    vec = np.full(len(labels), np.nan)
    for stage, val in STAGE_MAP.items():
        vec[np.asarray(labels) == stage] = val
    return vec


def load_hypno_labels(path):
    """Load a hypnogram .txt (one label per line). Returns (labels_array, warning_or_None).
    Tolerates MT (movement time). Other non-AASM labels trigger a warning but are kept."""
    labels = np.loadtxt(str(path), dtype=str).astype('<U10')
    labels = np.atleast_1d(labels)
    tolerated = {'MT'}
    unknown = sorted({s for s in labels if s not in STAGE_MAP and s not in tolerated})
    warning = None
    if unknown:
        n_unknown = int(np.isin(labels, unknown).sum())
        pct = n_unknown / len(labels) * 100
        warning = (f'{n_unknown}/{len(labels)} epochs ({pct:.1f}%) carry non-AASM label(s): '
                   f'{", ".join(unknown)}. Run remap_hypno to convert to W/N1/N2/N3/R.')
    return labels, warning


def load_events_csv(edf_path):
    """Load the Compumedics *_event_xml.csv next to the EDF if present.
    Returns a DataFrame with Name/Start/Duration (seconds), or None."""
    cand = Path(edf_path).with_name(f'{Path(edf_path).stem}_event_xml.csv')
    if not cand.exists():
        return None
    try:
        df = pd.read_csv(str(cand))
        cols = {c.lower(): c for c in df.columns}
        if 'name' in cols and 'start' in cols and 'duration' in cols:
            return df.rename(columns={cols['name']: 'Name', cols['start']: 'Start',
                                      cols['duration']: 'Duration'})[['Name', 'Start', 'Duration']]
    except Exception:
        return None
    return None


# Colour family for event overlays (matched by substring, case-insensitive).
EVENT_COLORS = [
    ('apnea',   '#c0392b'), ('hypopnea', '#e67e22'), ('arousal',  '#f1c40f'),
    ('desat',   '#2980b9'), ('limb',     '#27ae60'), ('movement', '#27ae60'),
]


def event_color(name):
    low = str(name).lower()
    for key, col in EVENT_COLORS:
        if key in low:
            return col
    return '#8e44ad'  # default (other)


def cur_stage():
    """Stage label of the current epoch from the working (rescored) hypnogram, or '?'."""
    lab = STATE['rescored_labels']
    i = STATE['cur_epoch']
    if lab is not None and 0 <= i < len(lab):
        return str(lab[i])
    return '?'

## Section 1 — Load a file

Pick an **EDF** (required), a **hypnogram** `.txt` (optional), and optionally a **remap/reref JSON** config.
The EDF header is parsed (no signal loaded yet) to list channels and pre-fill the EEG / EOG / EMG selections.
Click **Load file** to read the signal. Use the **Signal** toggle to switch every plot between the *raw* signal and a *preprocessed* copy (per-participant re-reference from the JSON + bandpass).

In [ ]:
# ----- Section 1 widgets -----
fc_edf = FileChooser(filter_pattern=['*.edf', '*.EDF'])
fc_edf.title = '<b>EDF file</b> (required):'
fc_hypno = FileChooser(filter_pattern='*.txt')
fc_hypno.title = '<b>Hypnogram .txt</b> (optional):'
fc_json = FileChooser(filter_pattern='*.json')
fc_json.title = '<b>remap_reref_persubject.json</b> (optional, matched by EDF filename):'

# Channel selection as checkboxes (auto pre-checked from detection). Rebuilt per EDF.
chan_cbs = {'EEG': {}, 'EOG': {}, 'EMG': {}}
box_eeg, box_eog, box_emg = widgets.VBox(), widgets.VBox(), widgets.VBox()
chan_accordion = widgets.Accordion(children=[box_eeg, box_eog, box_emg])
for _i, _t in enumerate(['EEG', 'EOG', 'EMG']):
    chan_accordion.set_title(_i, _t)
chan_accordion.selected_index = 0


def _build_channel_checkboxes(channels, types):
    """(Re)build one checkbox per channel under each type group; pre-check the detected type."""
    for grp, box in (('EEG', box_eeg), ('EOG', box_eog), ('EMG', box_emg)):
        cbs = {}
        for ch in channels:
            cbs[ch] = widgets.Checkbox(value=(types[ch] == grp), description=ch, indent=False,
                                       style={'description_width': 'initial'},
                                       layout=widgets.Layout(width='130px', margin='0'))
        chan_cbs[grp] = cbs
        box.children = [widgets.Box(list(cbs.values()),
                                    layout=widgets.Layout(flex_flow='row wrap', width='100%'))]


def _selected(grp):
    return [ch for ch, cb in chan_cbs[grp].items() if cb.value]


dd_reref = widgets.Dropdown(options=['none', 'average'], value='none',
                            description='Re-ref (no JSON):',
                            style={'description_width': 'initial'},
                            layout=widgets.Layout(width='280px'))
cb_filter = widgets.Checkbox(value=True, description='Bandpass filter (preprocessed)',
                             style={'description_width': 'initial'})
txt_lfreq = widgets.BoundedFloatText(value=0.1, min=0.0, max=100.0, step=0.05,
                                     description='l_freq (Hz):', style={'description_width': '90px'},
                                     layout=widgets.Layout(width='200px'))
txt_hfreq = widgets.BoundedFloatText(value=40.0, min=1.0, max=500.0, step=1.0,
                                     description='h_freq (Hz):', style={'description_width': '90px'},
                                     layout=widgets.Layout(width='200px'))

btn_load = widgets.Button(description='▶  Load file', button_style='primary',
                          layout=widgets.Layout(width='200px', height='40px'))
info_panel = widgets.HTML(value='')
out_load = widgets.Output()

_HDR = {'ch_sf': {}, 'matched_cfg': None}     # header-derived info cached between callbacks
on_load_callbacks = []                         # later sections append their reset fn here


def reset_sections(*_):
    """Reset every section to its 'press Run' state (called after a new file is loaded)."""
    for cb in on_load_callbacks:
        try:
            cb()
        except Exception as e:
            with out_load:
                print(f'⚠ reset error: {e}')


def _on_edf_selected(chooser):
    """Parse the header (no signal), classify channels, build the channel checkboxes."""
    out_load.clear_output()
    info_panel.value = ''
    if not fc_edf.selected:
        return
    edf_path = Path(fc_edf.selected)
    fc_hypno.reset(path=str(edf_path.parent))
    fc_json.reset(path=str(edf_path.parent))
    try:
        hdr = read_edf_header_custom(str(edf_path))
    except Exception as e:
        info_panel.value = f'<span style="color:#c0392b;">Could not read EDF header: {e}</span>'
        return
    channels = hdr['channel']
    transducers = hdr.get('transducer_type', [''] * len(channels))
    types = detect_channel_types(channels, transducers)
    _HDR['ch_sf'] = {ch: sf for ch, sf in zip(channels, hdr['sampling_frequency'])}
    _build_channel_checkboxes(channels, types)
    n_eeg = sum(types[c] == 'EEG' for c in channels)
    n_eog = sum(types[c] == 'EOG' for c in channels)
    n_emg = sum(types[c] == 'EMG' for c in channels)
    sf_set = sorted(set(_HDR['ch_sf'].values()))
    info_panel.value = (
        f'<b>{edf_path.name}</b> — {len(channels)} channels, '
        f'sampling rates: {", ".join(sf_set)} Hz<br>'
        f'<small>Auto-checked: {n_eeg} EEG · {n_eog} EOG · {n_emg} EMG '
        f'(tick/untick channels in the groups below if needed)</small>')
    _try_match_json()


def _try_match_json(chooser=None):
    """Match the JSON config entry to this EDF by filename stem (normcase-insensitive)."""
    _HDR['matched_cfg'] = None
    if not fc_edf.selected or not fc_json.selected:
        return
    stem = Path(fc_edf.selected).stem
    try:
        cfg = load_json_lenient(fc_json.selected)
    except Exception as e:
        info_panel.value += f'<br><span style="color:#c0392b;">JSON load error: {e}</span>'
        return
    key = next((k for k in cfg if os.path.normcase(k) == os.path.normcase(stem)), None)
    if key is None:
        info_panel.value += ('<br><small style="color:#e67e00;">No JSON entry matches '
                             f'<b>{stem}</b> — the EEG checkboxes + manual re-ref will be used.</small>')
    else:
        _HDR['matched_cfg'] = cfg[key]
        remap = cfg[key].get('remap', {})
        ref = cfg[key].get('ref_channels', [])
        info_panel.value += ('<br><small style="color:#2e7d32;">JSON entry matched: '
                             f'{len(remap)} EEG channel(s), ref = {ref}.</small>')


fc_edf.register_callback(_on_edf_selected)
fc_json.register_callback(_try_match_json)


def on_load(btn):
    btn.disabled = True
    out_load.clear_output()
    try:
        if not fc_edf.selected:
            with out_load:
                print('ERROR: select an EDF file first.')
            return
        edf_path = Path(fc_edf.selected)
        cfg = _HDR['matched_cfg']

        # EEG analysis set: remap keys if a JSON entry matched, else the checked EEG boxes.
        if cfg is not None:
            eeg_orig = list(cfg.get('remap', {}).keys())
        else:
            eeg_orig = _selected('EEG')
        eog_orig = _selected('EOG')
        emg_orig = _selected('EMG')
        if not eeg_orig:
            with out_load:
                print('ERROR: no EEG channels selected / found.')
            return

        # include= union at read time (original names) — preserves the EEG native rate and
        # avoids the MNE partial-read AssertionError (see SPEC Phase 2 note).
        include = list(dict.fromkeys(eeg_orig + eog_orig + emg_orig))
        with out_load:
            print(f'Loading {edf_path.name} … ({len(include)} channels)')
        raw = mne.io.read_raw_edf(str(edf_path), preload=True, encoding='latin-1',
                                  include=include, verbose=False)

        # Physical bounds (µV) per channel, captured before any renaming.
        extras = raw._raw_extras[0]
        phys_by_name = {}
        for idx, ch in enumerate(raw.ch_names):
            base = ch.rsplit('-', 1)[0] if (ch.endswith('-0') or ch.endswith('-1')) else ch
            try:
                phys_by_name[base] = get_phys_bounds_uV(extras, idx)
            except Exception:
                phys_by_name[base] = (np.nan, np.nan)

        raw, _ = drop_suffix_duplicates(raw)

        # Rename EEG to harmonised names when a JSON entry matched; EOG/EMG keep original names.
        final_to_orig = {}
        if cfg is not None:
            remap_adapted = adapt_remap_dict_to_suffixes(raw, cfg.get('remap', {}))
            for orig, new in remap_adapted.items():
                base = orig.rsplit('-', 1)[0] if (orig.endswith('-0') or orig.endswith('-1')) else orig
                final_to_orig[new] = base
            raw.rename_channels(remap_adapted)
            eeg_channels = [remap_adapted[o] for o in remap_adapted]
        else:
            eeg_channels = [c for c in eeg_orig if c in raw.ch_names]
            final_to_orig = {c: c for c in eeg_channels}
        eog_channels = [c for c in eog_orig if c in raw.ch_names]
        emg_channels = [c for c in emg_orig if c in raw.ch_names]

        # Tag channel types (lets set_eeg_reference touch only EEG, and labels EOG/EMG).
        type_map = {**{c: 'eeg' for c in eeg_channels},
                    **{c: 'eog' for c in eog_channels},
                    **{c: 'emg' for c in emg_channels}}
        try:
            raw.set_channel_types({c: t for c, t in type_map.items() if c in raw.ch_names})
        except Exception:
            pass

        # Resampling (Q1): bring everything to the EEG native rate so the scoring montage
        # shares one time base. MNE reads the union at the highest included rate; resample back.
        try:
            eeg_sf = float(_HDR['ch_sf'].get(final_to_orig.get(eeg_channels[0], eeg_channels[0]),
                                             raw.info['sfreq']))
        except Exception:
            eeg_sf = float(raw.info['sfreq'])
        if abs(raw.info['sfreq'] - eeg_sf) > 1e-3:
            with out_load:
                print(f'Resampling {raw.info["sfreq"]:.0f} Hz → {eeg_sf:.0f} Hz (EEG native rate) …')
            raw.resample(eeg_sf, npad='auto', verbose=False)
        sf = float(raw.info['sfreq'])

        # Preprocessed copy: re-reference (JSON ref or manual) + optional bandpass. Reference
        # channels are NOT dropped here so the channel set stays identical to the raw copy.
        raw_proc = raw.copy()
        ref = cfg.get('ref_channels', []) if cfg is not None else None
        try:
            if ref == 'average' or (cfg is None and dd_reref.value == 'average'):
                raw_proc.set_eeg_reference('average', verbose=False)
            elif isinstance(ref, list) and ref:
                ref_present = [c for c in ref if c in raw_proc.ch_names]
                if ref_present:
                    raw_proc.set_eeg_reference(ref_channels=ref_present, verbose=False)
        except Exception as e:
            with out_load:
                print(f'⚠ re-reference skipped: {e}')
        if cb_filter.value:
            try:
                raw_proc.filter(l_freq=txt_lfreq.value, h_freq=txt_hfreq.value, method='fir',
                                phase='zero-double', fir_window='hamming', fir_design='firwin',
                                verbose=False)
            except Exception as e:
                with out_load:
                    print(f'⚠ bandpass filter skipped: {e}')

        n_epochs = int(np.floor(raw.n_times / sf / 30))

        # Hypnogram (optional): align length to the signal epochs.
        hypno_labels = None
        hypno_warn = None
        if fc_hypno.selected:
            try:
                labels, hypno_warn = load_hypno_labels(fc_hypno.selected)
                if len(labels) != n_epochs:
                    hypno_warn = ((hypno_warn + ' ') if hypno_warn else '') + (
                        f'Length mismatch: signal has {n_epochs} epochs, hypnogram has {len(labels)} '
                        '— aligned by truncation/padding with "?".')
                    if len(labels) > n_epochs:
                        labels = labels[:n_epochs]
                    else:
                        labels = np.concatenate([labels, np.array(['?'] * (n_epochs - len(labels)))])
                hypno_labels = labels
            except Exception as e:
                hypno_warn = f'Hypnogram load failed: {e}'

        # Commit to shared state.
        STATE.update({
            'edf_path': edf_path, 'hypno_path': Path(fc_hypno.selected) if fc_hypno.selected else None,
            'raw_raw': raw, 'raw_proc': raw_proc,
            'eeg_channels': eeg_channels, 'eog_channels': eog_channels, 'emg_channels': emg_channels,
            'sf': sf, 'n_epochs': n_epochs,
            'hypno_labels': hypno_labels,
            'rescored_labels': hypno_labels.copy() if hypno_labels is not None else None,
            'events': load_events_csv(edf_path),
            'phys_bounds': {new: phys_by_name.get(orig, (np.nan, np.nan))
                            for new, orig in final_to_orig.items()},
            'final_to_orig': final_to_orig, 'reject_masks': None, 'cur_epoch': 0,
        })

        with out_load:
            ev = STATE['events']
            print(f'Loaded: {len(eeg_channels)} EEG, {len(eog_channels)} EOG, {len(emg_channels)} EMG '
                  f'@ {sf:.0f} Hz · {n_epochs} epochs ({n_epochs * 30 / 3600:.1f} h)')
            print(f'Hypnogram: {"loaded" if hypno_labels is not None else "none"}'
                  + (f' · {len(ev)} events' if ev is not None else ' · no event file'))
            if hypno_warn:
                print(f'⚠ {hypno_warn}')
            print('→ Press the Run buttons in Sections 2 and 3 to draw their plots.')
        reset_sections()
    except Exception as e:
        with out_load:
            print(f'Unexpected error while loading: {repr(e)}')
    finally:
        btn.disabled = False


btn_load.on_click(on_load)

display(
    HTML('<h4>Files</h4>'), fc_edf, fc_hypno, fc_json,
    HTML('<h4>Channels (tick to include; auto-checked from detection)</h4>'), chan_accordion,
    HTML('<h4>Preprocessed-view options</h4>'),
    widgets.HBox([dd_reref, cb_filter, txt_lfreq, txt_hfreq]),
    HTML('<small>When a JSON entry matches, its EEG remap + re-reference are used and the EEG '
         'checkboxes are ignored. Each section below has its own Raw/Preprocessed selector.</small>'),
    btn_load, info_panel, out_load,
)

## Section 2 — Whole-recording overview (EEG)

Per-electrode quality view, computed on the signal selected by the **Signal** toggle above.
Pick an electrode to see its amplitude histogram, full time series, quality metrics, hypnospectrogram,
whole-night PSD, and mean PSD per sleep stage.

In [ ]:
# ----- Section 2 widgets -----
sig2 = make_signal_toggle()
dd_elec = widgets.Dropdown(description='Electrode:', options=[],
                           style={'description_width': 'initial'},
                           layout=widgets.Layout(width='260px'))
btn_run_sec2 = widgets.Button(description='▶ Run overview', button_style='primary',
                              layout=widgets.Layout(width='190px', height='36px'))
sec2_info = widgets.HTML(value='')
out_sec2 = widgets.Output()

# Default flagging thresholds (same as quality_overview) for the metrics-table flags.
SEC2_THRESHOLDS = {'flat_pct': 3.5, 'bounds_pct': 1.0, 'n_peaks': 2, 'std_low': 5.0, 'hist_extreme_pct': 1.0}
# Precomputed per-source, per-electrode content, so switching electrode AND Raw/Preprocessed is instant.
SEC2_CACHE = {'by_source': {}}   # {'Raw': {ch: content}, 'Preprocessed': {ch: content}}


def _metrics_table_html(m, flags):
    flag_bg = '#f8d7da' if flags else '#e8f5e9'
    flag_txt = '; '.join(flags) if flags else 'no flags'
    rows = [
        ('Mean', f"{m['mean_uV']:.2f} µV"), ('Std dev', f"{m['std_uV']:.2f} µV"),
        ('p99 |amp|', f"{m['p99_abs_uV']:.1f} µV"), ('p99.9 |amp|', f"{m['p999_abs_uV']:.1f} µV"),
        ('Kurtosis', f"{m['kurtosis']:.1f}"), ('Skewness', f"{m['skewness']:.3f}"),
        ('Flat (%)', f"{m['flat_pct']:.2f}%"), ('At EDF bounds (%)', f"{m['bounds_pct']:.3f}%"),
        ('Extreme histogram (%)', f"{m['hist_extreme_pct']:.3f}%"), ('n_peaks', str(m['n_peaks'])),
    ]
    body = ''.join(f'<tr><td style="padding:3px 10px;">{k}</td>'
                   f'<td style="padding:3px 10px;text-align:right;">{v}</td></tr>' for k, v in rows)
    return (f'<div style="background:{flag_bg};padding:6px;border-radius:4px;margin-bottom:6px;">'
            f'<b>Flags:</b> {flag_txt}</div>'
            '<table style="border-collapse:collapse;font-size:0.9em;">'
            '<tr style="background:#eee;"><th style="padding:3px 10px;text-align:left;">Metric</th>'
            '<th style="padding:3px 10px;">Value</th></tr>' + body + '</table>')


def _compute_electrode_content(ch, raw, sf, labels):
    """Build the metrics HTML + PNGs for one electrode (rendered once, cached)."""
    sig = raw.get_data(picks=[ch])[0] * 1e6
    pmin, pmax = STATE['phys_bounds'].get(ch, (np.nan, np.nan))
    m = compute_signal_metrics(sig, pmin, pmax)
    flags = flag_channel(m, SEC2_THRESHOLDS)
    x_lim = float(np.percentile(np.abs(sig), 99.9))
    pngs = [fig_to_png_bytes(plot_histogram_figure(m, ch, x_lim=x_lim)),
            fig_to_png_bytes(plot_timeseries_figure(sig, sf, x_lim, title=f'{ch} — time series')),
            fig_to_png_bytes(plot_psd_figure(sig, sf, title=f'{ch} — whole-night PSD'))]
    notes = []
    if labels is not None and np.isfinite(labels_to_yasa_vec(labels)).any():
        try:
            pngs.append(fig_to_png_bytes(make_spectrogram_figure(sig, sf, labels_to_yasa_vec(labels))))
        except Exception as e:
            notes.append(f'Spectrogram failed: {e}')
        try:
            pngs.append(fig_to_png_bytes(plot_perstage_psd_figure(sig, sf, labels, ch)))
        except Exception as e:
            notes.append(f'Per-stage PSD failed: {e}')
    else:
        notes.append('No hypnogram — spectrogram and per-stage PSD skipped.')
    return {'metrics_html': _metrics_table_html(m, flags), 'pngs': pngs, 'notes': notes}


def _display_cached(ch):
    src = sig2.value
    content = SEC2_CACHE['by_source'].get(src, {}).get(ch)
    with out_sec2:
        clear_output(wait=True)
        if content is None:
            print('Press "Run overview" first.')
            return
        display(HTML(f'<b>{ch}</b> &mdash; signal source: <i>{src}</i> '
                     '<small>(toggle Raw/Preprocessed for instant comparison)</small>'))
        display(HTML(content['metrics_html']))
        for png in content['pngs']:
            display(widgets.Image(value=png, format='png'))
        for note in content['notes']:
            display(HTML(f'<small style="color:#888;">⚠ {note}</small>'))


def on_run_sec2(btn):
    btn.disabled = True
    try:
        if STATE['raw_raw'] is None or not STATE['eeg_channels']:
            sec2_info.value = '<span style="color:#c0392b;">Load a file in Section 1 first.</span>'
            return
        eeg = STATE['eeg_channels']
        labels = STATE['rescored_labels']
        sf = STATE['sf']
        SEC2_CACHE['by_source'] = {}
        # Precompute both signal sources so the Raw/Preprocessed toggle is instant.
        sources = [s for s in ('Raw', 'Preprocessed') if raw_for(s) is not None]
        for src in sources:
            raw = raw_for(src)
            SEC2_CACHE['by_source'][src] = {}
            for i, ch in enumerate(eeg):
                sec2_info.value = f'<small>Computing {src} — {ch} ({i + 1}/{len(eeg)}) …</small>'
                SEC2_CACHE['by_source'][src][ch] = _compute_electrode_content(ch, raw, sf, labels)
        sec2_info.value = (f'<small style="color:#2e7d32;">Ready — {len(eeg)} electrode(s) computed for '
                           f'{" & ".join(sources)}. Switch electrode and Raw/Preprocessed instantly below.</small>')
        if dd_elec.value not in eeg:
            dd_elec.value = eeg[0]
        _display_cached(dd_elec.value)
    except Exception as e:
        sec2_info.value = f'<span style="color:#c0392b;">Error: {repr(e)}</span>'
    finally:
        btn.disabled = False


def _on_elec_change(change):
    _display_cached(change['new'])


def _on_sig2_change(change):
    # Instant switch if already computed; otherwise prompt to Run.
    if SEC2_CACHE['by_source'].get(sig2.value):
        _display_cached(dd_elec.value)
    else:
        with out_sec2:
            clear_output(wait=True)
            print('Press "Run overview" to compute this signal source.')


def reset_sec2(*_):
    """Sync the electrode list and invalidate the cache (Run again to recompute)."""
    SEC2_CACHE['by_source'] = {}
    out_sec2.clear_output()
    if STATE['raw_raw'] is None:
        sec2_info.value = ''
        return
    dd_elec.unobserve(_on_elec_change, names='value')
    dd_elec.options = STATE['eeg_channels']
    if STATE['eeg_channels']:
        dd_elec.value = STATE['eeg_channels'][0]
    dd_elec.observe(_on_elec_change, names='value')
    sec2_info.value = ('<small style="color:#e67e00;">Press "Run overview" to compute the plots for '
                       'all electrodes (both Raw and Preprocessed, for instant comparison).</small>')


dd_elec.observe(_on_elec_change, names='value')
btn_run_sec2.on_click(on_run_sec2)
sig2.observe(_on_sig2_change, names='value')   # instant switch between cached sources
on_load_callbacks.append(reset_sec2)

display(widgets.HBox([btn_run_sec2, sig2]), widgets.HBox([dd_elec]), sec2_info, out_sec2)

## Section 3 — Epoch explorer (EEG + EOG + EMG)

Navigate the recording one 30 s epoch at a time. The **navigator** shows the whole-night hypnogram
(with a cursor and event ticks) under a cached spectrogram of a reference channel. The **epoch view**
stacks the scoring montage (EEG + EOG + EMG) with event annotations, plus the per-epoch PSD.

**Rescoring**: click a stage button to reassign the current epoch (held in memory). Modified epochs are
ticked on the navigator. **Save rescored hypnogram** writes a new `_rescored.txt` next to the input
hypnogram — the original is never overwritten. Cross-check edits in your certified scoring software.

In [ ]:
# ----- Section 3 widgets -----
sig3 = make_signal_toggle()
btn_run_sec3 = widgets.Button(description='▶ Run explorer', button_style='primary',
                              layout=widgets.Layout(width='180px', height='36px'))
dd_ref = widgets.Dropdown(description='Spectrogram ref:', options=[],
                          style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'))
sel_disp = widgets.SelectMultiple(description='Show EEG:', options=[], rows=5,
                                  style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'))
sl_epoch = widgets.IntSlider(description='Epoch:', min=0, max=0, value=0, continuous_update=False,
                             style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))
btn_prev = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
btn_next_stage = widgets.Button(description='Next stage ⇥', layout=widgets.Layout(width='130px'))
dd_ctx = widgets.Dropdown(description='Context:', options=[('single epoch', 0), ('±1', 1), ('±2', 2)],
                          value=0, style={'description_width': 'initial'}, layout=widgets.Layout(width='200px'))
sl_amp = widgets.FloatSlider(description='Scale (µV):', min=20, max=600, step=10, value=150,
                             continuous_update=False, style={'description_width': 'initial'},
                             layout=widgets.Layout(width='320px'))
stage_buttons = [widgets.Button(description=s, layout=widgets.Layout(width='52px'),
                                button_style='warning') for s in ['W', 'N1', 'N2', 'N3', 'R', 'MT', '?']]
cb_autoadv = widgets.Checkbox(value=True, description='Auto-advance after scoring',
                              style={'description_width': 'initial'})
cb_overwrite = widgets.Checkbox(value=False, description='Overwrite existing _rescored',
                                style={'description_width': 'initial'})
btn_save_hypno = widgets.Button(description='💾 Save rescored hypnogram', button_style='success',
                                layout=widgets.Layout(width='260px'))
lbl_status = widgets.HTML(value='')
out_nav = widgets.Output()
out_epoch = widgets.Output()
out_save = widgets.Output()

# Navigator cache: the spectrogram arrays are computed once on Run, for BOTH signal sources, so the
# Raw/Preprocessed toggle switches instantly. Per-step redraws (cursor move) are then cheap.
_NAV = {'ready': False, 'spec_by_source': {}, 'ref': None}

# Keyboard shortcuts (when ipyevents is available): arrows step epochs, letters/digits score.
KEY_TO_STAGE = {'w': 'W', '0': 'W', '1': 'N1', '2': 'N2', '3': 'N3',
                '4': 'R', '5': 'R', 'r': 'R', 'm': 'MT'}


def compute_nav_spectrogram(sig_uV, sf):
    """Spectrogram arrays (x in epoch units, freq <= 30 Hz, power in dB) for the navigator.
    Uses 30 s windows (50% overlap) so the time axis is epoch-scaled and redraws stay fast."""
    nper = max(64, int(sf * 30))           # 30 s windows, epoch-aligned
    f, t, Sxx = scipy_spectrogram(sig_uV, fs=sf, nperseg=nper, noverlap=nper // 2)
    mask = f <= 30
    Sxx_db = 10 * np.log10(np.maximum(Sxx[mask], 1e-12))
    return t / 30.0, f[mask], Sxx_db


def plot_navigator(cur):
    """Transparent spectrogram over the hypnogram (twin axes), bold cursor + enlarged markers."""
    n = STATE['n_epochs']
    labels = STATE['rescored_labels']
    fig, ax = plt.subplots(figsize=(13, 2.8))
    spec = _NAV['spec_by_source'].get(sig3.value)
    if spec is not None:
        t_ep, f, Sxx_db = spec
        # Robust colour scaling: clip to p5-p99 so the silent floor (-120 dB) and artefact
        # spikes (+70 dB) don't compress the meaningful ~40 dB band into flat mid-colours.
        vmin, vmax = float(np.percentile(Sxx_db, 5)), float(np.percentile(Sxx_db, 99))
        ax.pcolormesh(t_ep, f, Sxx_db, cmap='viridis', vmin=vmin, vmax=vmax,
                      alpha=0.9, shading='auto', rasterized=True)
        ax.set_ylabel('Freq (Hz)', fontsize=8)
        ax.set_ylim(0.5, 30)
        ax.tick_params(axis='y', labelsize=7)
    ax.set_xlim(0, n)
    ax.set_xlabel('Epoch index', fontsize=8)
    # Hypnogram on a twin axis so its stage scale stays readable over the spectrogram.
    ax2 = ax.twinx()
    STAGE_H = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    if labels is not None:
        y = [STAGE_H.get(s, -0.6) for s in labels]
        ax2.step(np.arange(n + 1), y + [y[-1]], where='post', color='white', linewidth=1.7, zorder=6,
                 path_effects=[mpe.Stroke(linewidth=3.4, foreground='black'), mpe.Normal()])
        ax2.set_yticks([0, 1, 2, 3, 4])
        ax2.set_yticklabels(['N3', 'N2', 'N1', 'REM', 'W'], fontsize=8)
        orig = STATE['hypno_labels']
        if orig is not None:
            for ei in np.where(np.asarray(labels) != np.asarray(orig))[0]:
                ax2.plot([ei + 0.5], [4.7], marker='v', color='#e67e22', markersize=8, zorder=8)
    else:
        ax2.set_yticks([])
    ax2.set_ylim(-1.3, 5.0)
    ax2.set_xlim(0, n)
    # Event ticks along the bottom (enlarged, colour-coded).
    ev = STATE['events']
    if ev is not None and len(ev):
        for _, row in ev.iterrows():
            ax2.plot([row['Start'] / 30.0], [-0.9], marker='|', color=event_color(row['Name']),
                     markersize=12, markeredgewidth=2.5, zorder=7,
                     path_effects=[mpe.Stroke(linewidth=4, foreground='black'), mpe.Normal()])
    # Cursor at the current epoch (bold, high contrast).
    ax2.axvline(cur + 0.5, color='#ff1744', linewidth=2.5, zorder=10)
    for sp in ['top']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


def plot_epoch_view(cur):
    """Stacked time series of the scoring montage for the current epoch (+context), with events."""
    raw = raw_for(sig3.value)
    sf = STATE['sf']
    ctx = dd_ctx.value
    disp_eeg = [c for c in sel_disp.value] or STATE['eeg_channels']
    chans = list(disp_eeg) + STATE['eog_channels'] + STATE['emg_channels']
    e0 = max(0, cur - ctx)
    e1 = min(STATE['n_epochs'], cur + ctx + 1)
    s0 = int(e0 * 30 * sf)
    s1 = int(e1 * 30 * sf)
    t = np.arange(s0, s1) / sf
    data = raw.get_data(picks=chans, start=s0, stop=s1) * 1e6
    gap = sl_amp.value
    offsets = gap * np.arange(len(chans))[::-1]
    fig, ax = plt.subplots(figsize=(12, 0.55 * len(chans) + 1.5))
    ax.axvspan(cur * 30, (cur + 1) * 30, color='#000000', alpha=0.05, zorder=0)
    for e in range(e0, e1 + 1):
        ax.axvline(e * 30, color='#cccccc', linewidth=0.6, linestyle='--', zorder=1)
    for i, ch in enumerate(chans):
        sig = data[i] - np.mean(data[i])
        ax.plot(t, sig + offsets[i], linewidth=0.5, color='#2a3f5f', zorder=3)
    ev = STATE['events']
    active_types = {}
    if ev is not None and len(ev):
        for _, row in ev.iterrows():
            es, ee = row['Start'], row['Start'] + row['Duration']
            if ee >= t[0] and es <= t[-1]:
                col = event_color(row['Name'])
                ax.axvspan(max(es, t[0]), min(ee, t[-1]), color=col, alpha=0.15, zorder=2)
                active_types[str(row['Name'])] = col
    ax.set_yticks(offsets)
    ax.set_yticklabels(chans, fontsize=8)
    ax.set_xlim(t[0], t[-1])
    ax.set_ylim(offsets.min() - gap, offsets.max() + gap)
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.set_title(f'Epoch {cur} — stage {cur_stage()} — {sig3.value} — scale {gap:.0f} µV/trace', fontsize=10)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    if active_types:
        handles = [matplotlib.patches.Patch(color=c, alpha=0.4, label=n) for n, c in active_types.items()]
        ax.legend(handles=handles, fontsize=7, loc='upper right', ncol=2)
    plt.tight_layout()
    return fig


def plot_epoch_psd(cur):
    """Per-epoch Welch PSD overlay for the displayed EEG channels."""
    raw = raw_for(sig3.value)
    sf = STATE['sf']
    disp_eeg = [c for c in sel_disp.value] or STATE['eeg_channels']
    s0 = int(cur * 30 * sf)
    s1 = int((cur + 1) * 30 * sf)
    fig, ax = plt.subplots(figsize=(7, 3))
    _annotate_bands(ax, min(40.0, sf / 2 - 0.5))
    readout = []
    for ch in disp_eeg:
        sig = raw.get_data(picks=[ch], start=s0, stop=s1)[0] * 1e6
        freqs, psd = welch_psd(sig, sf, fmin=0.5, fmax=min(40.0, sf / 2 - 0.5))
        ax.plot(freqs, 10 * np.log10(np.maximum(psd, 1e-10)), linewidth=1.0, label=ch)
        readout.append(f'{ch}: p-p {np.ptp(sig):.0f} µV, max|Δ| {np.max(np.abs(np.diff(sig))):.0f} µV')
    ax.set_xlim(0.5, min(40.0, sf / 2 - 0.5))
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB/Hz)', fontsize=11)
    ax.set_title(f'Epoch {cur} — PSD ({sig3.value})', fontsize=11)
    ax.legend(fontsize=7, ncol=2)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig, readout


def redraw_nav():
    if not _NAV['ready']:
        return
    show_fig(plot_navigator(STATE['cur_epoch']), out_nav)


def redraw_epoch():
    if not _NAV['ready']:
        with out_epoch:
            clear_output(wait=True)
            print('Press "Run explorer" first.')
        return
    cur = STATE['cur_epoch']
    fig_ts = plot_epoch_view(cur)
    fig_psd, readout = plot_epoch_psd(cur)
    with out_epoch:
        clear_output(wait=True)
        display(widgets.Image(value=fig_to_png_bytes(fig_ts), format='png'))
        display(widgets.Image(value=fig_to_png_bytes(fig_psd), format='png'))
        display(HTML('<small>' + ' &nbsp;|&nbsp; '.join(readout) + '</small>'))
    _update_status()


def _update_status():
    cur = STATE['cur_epoch']
    n = STATE['n_epochs']
    t = cur * 30
    hh, mm, ss = t // 3600, (t % 3600) // 60, t % 60
    n_mod = 0
    if STATE['rescored_labels'] is not None and STATE['hypno_labels'] is not None:
        n_mod = int((np.asarray(STATE['rescored_labels']) != np.asarray(STATE['hypno_labels'])).sum())
    lbl_status.value = (f'<b>Epoch {cur}/{max(0, n - 1)}</b> &nbsp; t={hh:02d}:{mm:02d}:{ss:02d} '
                        f'&nbsp; stage <b>{cur_stage()}</b> &nbsp;·&nbsp; {n_mod} epoch(s) modified')


def on_run_sec3(btn):
    btn.disabled = True
    try:
        if STATE['raw_raw'] is None or not STATE['eeg_channels']:
            with out_epoch:
                clear_output(wait=True)
                print('Load a file in Section 1 first.')
            return
        ref = dd_ref.value or STATE['eeg_channels'][0]
        # Precompute the navigator spectrogram for BOTH signal sources so the toggle is instant.
        _NAV['spec_by_source'] = {}
        for src in [s for s in ('Raw', 'Preprocessed') if raw_for(s) is not None]:
            with out_nav:
                clear_output(wait=True)
                print(f'Computing navigator spectrogram ({src}) …')
            try:
                sig = raw_for(src).get_data(picks=[ref])[0] * 1e6
                _NAV['spec_by_source'][src] = compute_nav_spectrogram(sig, STATE['sf'])
            except Exception as e:
                _NAV['spec_by_source'][src] = None
                with out_nav:
                    print(f'⚠ spectrogram failed ({src}): {e}')
        _NAV['ref'] = ref
        _NAV['ready'] = True
        STATE['cur_epoch'] = 0
        sl_epoch.unobserve(_on_slider, names='value')
        sl_epoch.max = max(0, STATE['n_epochs'] - 1)
        sl_epoch.value = 0
        sl_epoch.observe(_on_slider, names='value')
        redraw_nav()
        redraw_epoch()
    except Exception as e:
        with out_epoch:
            print(f'Unexpected error: {repr(e)}')
    finally:
        btn.disabled = False


def reset_sec3(*_):
    """Sync widget options to the loaded file; require a Run press before drawing."""
    _NAV['ready'] = False
    _NAV['spec_by_source'] = {}
    out_nav.clear_output()
    out_epoch.clear_output()
    if STATE['raw_raw'] is None:
        return
    eeg = STATE['eeg_channels']
    dd_ref.options = eeg
    if dd_ref.value not in eeg and eeg:
        dd_ref.value = eeg[0]
    sel_disp.options = eeg
    sel_disp.value = tuple(eeg)
    STATE['cur_epoch'] = 0
    with out_nav:
        clear_output(wait=True)
        print('Press "Run explorer" to build the navigator and epoch view.')


def _on_ref_change(change):
    # The spectrogram is ref-specific, so changing it needs a Run to recompute.
    reset_sec3()


def _on_sig3_change(change):
    # Both sources are cached on Run, so the toggle switches instantly (nav from cache, epoch live).
    if _NAV['ready']:
        redraw_nav()
        redraw_epoch()


def goto_epoch(i):
    i = int(np.clip(i, 0, max(0, STATE['n_epochs'] - 1)))
    STATE['cur_epoch'] = i
    if sl_epoch.value != i:
        sl_epoch.value = i          # observer triggers the redraw
    else:
        redraw_nav()
        redraw_epoch()


def _on_slider(change):
    STATE['cur_epoch'] = int(change['new'])
    try:
        redraw_nav()
        redraw_epoch()
    except Exception as e:
        with out_epoch:
            print(f'⚠ {e}')


def _next_stage_change(btn):
    labels = STATE['rescored_labels']
    cur = STATE['cur_epoch']
    if labels is None:
        goto_epoch(cur + 1)
        return
    for j in range(cur + 1, len(labels)):
        if labels[j] != labels[cur]:
            goto_epoch(j)
            return
    goto_epoch(len(labels) - 1)


def score_current(stage):
    """Assign a stage to the current epoch (used by stage buttons and keyboard)."""
    if not _NAV['ready']:
        return
    if STATE['rescored_labels'] is None:
        STATE['rescored_labels'] = np.array(['?'] * STATE['n_epochs'], dtype='<U10')
        STATE['hypno_labels'] = STATE['rescored_labels'].copy()
    STATE['rescored_labels'][STATE['cur_epoch']] = stage
    if cb_autoadv.value and STATE['cur_epoch'] < STATE['n_epochs'] - 1:
        goto_epoch(STATE['cur_epoch'] + 1)
    else:
        redraw_nav()
        redraw_epoch()


def on_save_hypno(btn):
    out_save.clear_output()
    with out_save:
        if STATE['rescored_labels'] is None:
            print('Nothing to save — no hypnogram/scoring in memory.')
            return
        base = STATE['hypno_path'] if STATE['hypno_path'] is not None else STATE['edf_path']
        out_txt = base.with_name(f'{base.stem}_rescored.txt')
        out_log = base.with_name(f'{base.stem}_rescore_log.tsv')
        if out_txt.exists() and not cb_overwrite.value:
            print(f'⚠ {out_txt.name} already exists. Tick "Overwrite existing _rescored" to replace it.')
            return
        try:
            np.savetxt(str(out_txt), STATE['rescored_labels'], fmt='%s')
            orig = STATE['hypno_labels']
            log_rows = []
            if orig is not None:
                for ei in np.where(np.asarray(STATE['rescored_labels']) != np.asarray(orig))[0]:
                    log_rows.append({'epoch_idx': int(ei), 'epoch_time_sec': int(ei * 30),
                                     'original_label': str(orig[ei]),
                                     'new_label': str(STATE['rescored_labels'][ei]),
                                     'timestamp': datetime.now().isoformat(timespec='seconds')})
            pd.DataFrame(log_rows).to_csv(str(out_log), sep='\t', index=False)
            print(f'Saved {out_txt.name} ({len(STATE["rescored_labels"])} epochs, '
                  f'{len(log_rows)} modified) and {out_log.name} next to the hypnogram.')
        except Exception as e:
            print(f'Save failed: {e}')


btn_run_sec3.on_click(on_run_sec3)
btn_prev.on_click(lambda b: goto_epoch(STATE['cur_epoch'] - 1))
btn_next.on_click(lambda b: goto_epoch(STATE['cur_epoch'] + 1))
btn_next_stage.on_click(_next_stage_change)
for b, s in zip(stage_buttons, ['W', 'N1', 'N2', 'N3', 'R', 'MT', '?']):
    b.on_click(lambda btn, st=s: score_current(st))
btn_save_hypno.on_click(on_save_hypno)
sl_epoch.observe(_on_slider, names='value')
dd_ref.observe(_on_ref_change, names='value')      # changing the ref needs a Run to rebuild
sig3.observe(_on_sig3_change, names='value')        # both sources cached -> instant switch
for w in (sel_disp, dd_ctx, sl_amp):
    w.observe(lambda c: redraw_epoch(), names='value')
on_load_callbacks.append(reset_sec3)

_kbd_hint = ('keyboard: ←/→ epochs · w/0=W · 1/2/3 · 4/5/r=R · m=MT (click the panel first)'
             if _HAS_IPYEVENTS else 'install ipyevents for keyboard shortcuts')
sec3_box = widgets.VBox([
    widgets.HBox([btn_run_sec3, sig3]),
    widgets.HBox([dd_ref, dd_ctx, sl_amp]),
    sel_disp,
    out_nav,
    widgets.HBox([btn_prev, sl_epoch, btn_next, btn_next_stage]),
    lbl_status,
    widgets.HTML(f'<b>Rescore current epoch:</b> <small>({_kbd_hint})</small>'),
    widgets.HBox(stage_buttons + [cb_autoadv]),
    out_epoch,
    widgets.HBox([btn_save_hypno, cb_overwrite]),
    out_save,
])

if _HAS_IPYEVENTS:
    _kb = _IpyEvent(source=sec3_box, watched_events=['keydown'], prevent_default_action=True)

    def _on_key(event):
        k = event.get('key', '')
        if k == 'ArrowRight':
            goto_epoch(STATE['cur_epoch'] + 1)
        elif k == 'ArrowLeft':
            goto_epoch(STATE['cur_epoch'] - 1)
        elif k.lower() in KEY_TO_STAGE:
            score_current(KEY_TO_STAGE[k.lower()])

    _kb.on_dom_event(_on_key)

display(sec3_box)

## Section 4 — Quick epoch rejection (EEG)

Compute the five rejection methods live on the EEG channels of the signal selected by the **Signal** toggle.
View the channels × epochs heatmap, browse rejected epochs (jumps the Section 3 explorer to them), and
compare the mean PSD per stage over **clean** vs **rejected** epochs. Export a rejection mask TSV
(same schema as *preprocessing_voila*) — this is standalone QC and does not write a `.fif`.

In [ ]:
# ----- Section 4 widgets -----
sig4 = make_signal_toggle()
_ws = {'description_width': '180px'}
_wl = widgets.Layout(width='320px')
rj_W = widgets.BoundedFloatText(value=250.0, min=1, max=5000, step=10, description='Amplitude W (µV):', style=_ws, layout=_wl)
rj_N1 = widgets.BoundedFloatText(value=250.0, min=1, max=5000, step=10, description='Amplitude N1 (µV):', style=_ws, layout=_wl)
rj_N2 = widgets.BoundedFloatText(value=300.0, min=1, max=5000, step=10, description='Amplitude N2 (µV):', style=_ws, layout=_wl)
rj_N3 = widgets.BoundedFloatText(value=300.0, min=1, max=5000, step=10, description='Amplitude N3 (µV):', style=_ws, layout=_wl)
rj_R = widgets.BoundedFloatText(value=250.0, min=1, max=5000, step=10, description='Amplitude REM (µV):', style=_ws, layout=_wl)
rj_flat = widgets.BoundedFloatText(value=1.0, min=0.01, max=100, step=0.1, description='Flat < (µV):', style=_ws, layout=_wl)
rj_grad = widgets.BoundedFloatText(value=100.0, min=1, max=5000, step=10, description='Gradient (µV/sample):', style=_ws, layout=_wl)
rj_1f_err = widgets.BoundedFloatText(value=0.15, min=0, max=1, step=0.01, description='1/f error >:', style=_ws, layout=_wl)
rj_1f_r2 = widgets.BoundedFloatText(value=0.85, min=0, max=1, step=0.01, description='1/f R² <:', style=_ws, layout=_wl)
cb_m_amp = widgets.Checkbox(value=True, description='amplitude', style={'description_width': 'initial'})
cb_m_flat = widgets.Checkbox(value=True, description='flat', style={'description_width': 'initial'})
cb_m_grad = widgets.Checkbox(value=True, description='gradient', style={'description_width': 'initial'})
cb_m_1f = widgets.Checkbox(value=False, description='1/f fit (slow)', style={'description_width': 'initial'})

dd_psd_ch = widgets.Dropdown(description='PSD channel:', options=[],
                             style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'))
dd_rej = widgets.Dropdown(description='Rejected epoch:', options=[],
                          style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
btn_compute = widgets.Button(description='▶ Compute rejection', button_style='primary',
                             layout=widgets.Layout(width='220px', height='38px'))
btn_save_mask = widgets.Button(description='💾 Save rejection mask', button_style='success',
                               layout=widgets.Layout(width='240px'))
out_hm = widgets.Output()
out_summary = widgets.Output()
out_rejpsd = widgets.Output()
out_rejview = widgets.Output()
out_mask = widgets.Output()


def _hypno_epochs_array():
    """Stage labels per epoch for the rejection summary (from rescored hypno, else all '?')."""
    if STATE['rescored_labels'] is not None:
        return np.asarray(STATE['rescored_labels'])
    return np.array(['?'] * STATE['n_epochs'], dtype='<U10')


def plot_clean_vs_rejected_psd(ch, reject_epoch):
    """Side-by-side mean PSD per stage over clean vs rejected epochs for one channel."""
    raw = raw_for(sig4.value)
    sf = STATE['sf']
    labels = _hypno_epochs_array()
    sig = raw.get_data(picks=[ch])[0] * 1e6
    n_epochs = STATE['n_epochs']
    freqs, psds = compute_epoch_psds(sig, sf, n_epochs)
    psds_db = 10 * np.log10(np.maximum(psds, 1e-10))
    colors = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba', 'N3': '#6a51a3', 'R': '#c994c7'}
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
    for ax, keep, ttl in [(axes[0], ~reject_epoch, 'clean'), (axes[1], reject_epoch, 'rejected')]:
        _annotate_bands(ax, float(freqs[-1]))
        for st in STAGES:
            sel = (labels == st) & keep
            if sel.sum() == 0:
                continue
            ax.plot(freqs, psds_db[sel].mean(axis=0), linewidth=1.3, color=colors[st],
                    label=f'{st} (n={int(sel.sum())})')
        ax.set_xlim(0.5, float(freqs[-1]))
        ax.set_xlabel('Frequency (Hz)', fontsize=10)
        ax.set_title(f'{ch} — {ttl} epochs', fontsize=10)
        ax.legend(fontsize=7, ncol=2)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
    axes[0].set_ylabel('Power (dB/Hz)', fontsize=10)
    plt.tight_layout()
    return fig


def _reject_epoch_vector():
    if STATE['reject_masks'] is None:
        return None
    re = np.zeros(STATE['n_epochs'], dtype=bool)
    for name in REJECT_METHODS:
        re |= STATE['reject_masks'][name].any(axis=1)
    return re


def on_compute(btn):
    btn.disabled = True
    for o in (out_hm, out_summary, out_rejpsd, out_rejview):
        o.clear_output()
    try:
        raw = raw_for(sig4.value)
        if raw is None or not STATE['eeg_channels']:
            with out_hm:
                print('Load a file in Section 1 first.')
            return
        sf = STATE['sf']
        n_epochs = STATE['n_epochs']
        eeg = STATE['eeg_channels']
        with out_hm:
            print('Computing epochs and rejection masks …')
        raw_eeg = raw.copy().pick(eeg)
        epochs = mne.make_fixed_length_epochs(raw_eeg, duration=30, preload=True, verbose=False)
        epochs = epochs[:n_epochs]
        data_uV = epochs.get_data() * 1e6
        hypno_epochs = _hypno_epochs_array()[:len(epochs)]

        psds_data_uV2 = psd_freqs = None
        if cb_m_1f.value and _HAS_SPECPARAM:
            try:
                n_per_seg = int(4 * sf)
                pobj = epochs.compute_psd(method='welch', fmin=0.5, fmax=min(30.0, sf / 2 - 0.5),
                                          n_fft=n_per_seg, n_overlap=n_per_seg // 2, n_per_seg=n_per_seg,
                                          window='hann', verbose=False)
                psds_data_uV2 = pobj.get_data() * 1e12
                psd_freqs = pobj.freqs
            except Exception as e:
                with out_hm:
                    print(f'⚠ PSD failed — 1/f rejection disabled: {e}')

        ptp_thresh = {'W': rj_W.value, 'N1': rj_N1.value, 'N2': rj_N2.value, 'N3': rj_N3.value, 'R': rj_R.value}
        masks = compute_rejection_masks(data_uV, sf, hypno_epochs, ptp_thresh,
                                        rj_flat.value, rj_grad.value, psd_freqs, psds_data_uV2,
                                        rj_1f_err.value, rj_1f_r2.value)
        enabled = {'amplitude': cb_m_amp.value, 'flat': cb_m_flat.value, 'gradient': cb_m_grad.value,
                   '1f_error': cb_m_1f.value, '1f_r2': cb_m_1f.value}
        for name in REJECT_METHODS:
            if not enabled[name]:
                masks[name][:] = False
        STATE['reject_masks'] = masks
        combined = build_combined_method_matrix(masks)
        reject_epoch = _reject_epoch_vector()

        fid = STATE['edf_path'].stem
        show_fig(plot_rejection_heatmap(combined, eeg, hypno_epochs, fid), out_hm)

        summary = build_rejection_summary(masks, hypno_epochs, fid)
        n_rej = int(reject_epoch.sum())
        with out_summary:
            clear_output(wait=True)
            display(HTML(f'<b>{n_rej}/{len(epochs)} epochs rejected ({100 * n_rej / max(1, len(epochs)):.1f}%)</b>'))
            any_rows = summary[summary['method'] == 'any'][['stage', 'n_total', 'n_rejected', 'pct_rejected']]
            display(HTML(any_rows.to_html(index=False, border=0)))

        # Clean-vs-rejected per-stage PSD (shown first).
        dd_psd_ch.options = eeg
        if dd_psd_ch.value not in eeg:
            dd_psd_ch.value = eeg[0]
        _draw_clean_vs_rejected()

        # Rejected-epoch inspector (populate dropdown; selecting one renders it below).
        rej_idx = np.where(reject_epoch)[0]
        dd_rej.options = [(f'epoch {i} ({hypno_epochs[i]})', int(i)) for i in rej_idx]
        with out_rejview:
            clear_output(wait=True)
            print('Select a rejected epoch above to inspect its montage and PSD.')
    except Exception as e:
        with out_hm:
            print(f'Unexpected error: {repr(e)}')
    finally:
        btn.disabled = False


def _draw_clean_vs_rejected():
    reject_epoch = _reject_epoch_vector()
    if reject_epoch is None:
        return
    if STATE['rescored_labels'] is not None and reject_epoch.any() and (~reject_epoch).any():
        try:
            show_fig(plot_clean_vs_rejected_psd(dd_psd_ch.value, reject_epoch), out_rejpsd)
        except Exception as e:
            with out_rejpsd:
                clear_output(wait=True)
                print(f'⚠ clean-vs-rejected PSD failed: {e}')
    else:
        with out_rejpsd:
            clear_output(wait=True)
            print('Clean-vs-rejected PSD needs a hypnogram and both clean & rejected epochs.')


def _on_psd_ch_change(change):
    if STATE['reject_masks'] is not None:
        _draw_clean_vs_rejected()


def _on_rej_select(change):
    """Render the selected rejected epoch's montage + PSD inline, and sync the Section 3 explorer."""
    if change['new'] is None:
        return
    idx = int(change['new'])
    try:
        fig_ts = plot_epoch_view(idx)
        fig_psd, readout = plot_epoch_psd(idx)
        with out_rejview:
            clear_output(wait=True)
            display(widgets.Image(value=fig_to_png_bytes(fig_ts), format='png'))
            display(widgets.Image(value=fig_to_png_bytes(fig_psd), format='png'))
            display(HTML('<small>' + ' &nbsp;|&nbsp; '.join(readout) + '</small>'))
    except Exception as e:
        with out_rejview:
            clear_output(wait=True)
            print(f'⚠ {e}')
    goto_epoch(idx)   # also move the Section 3 explorer there (if it has been run)


def on_save_mask(btn):
    out_mask.clear_output()
    with out_mask:
        if STATE['reject_masks'] is None:
            print('Compute rejection first.')
            return
        masks = STATE['reject_masks']
        eeg = STATE['eeg_channels']
        hypno_epochs = _hypno_epochs_array()
        rows = []
        n_epochs = masks['amplitude'].shape[0]
        for ei in range(n_epochs):
            for ci, ch in enumerate(eeg):
                row = {'epoch_idx': ei, 'stage': hypno_epochs[ei], 'channel': ch,
                       'reject_flag': bool(any(masks[m][ei, ci] for m in REJECT_METHODS))}
                for m in REJECT_METHODS:
                    row['flag_' + m.replace('/', '').replace(' ', '')] = bool(masks[m][ei, ci])
                rows.append(row)
        base = STATE['edf_path']
        out_path = base.with_name(f'{base.stem}_live_rejection_mask.tsv')
        try:
            pd.DataFrame(rows).to_csv(str(out_path), sep='\t', index=False)
            print(f'Saved {out_path.name} ({len(rows)} epoch×channel rows) next to the EDF.')
        except Exception as e:
            print(f'Save failed: {e}')


def reset_sec4(*_):
    STATE['reject_masks'] = None
    dd_rej.options = []
    if raw_for(sig4.value) is not None:
        dd_psd_ch.options = STATE['eeg_channels']
        if STATE['eeg_channels']:
            dd_psd_ch.value = STATE['eeg_channels'][0]
    for o in (out_hm, out_summary, out_rejpsd, out_rejview, out_mask):
        o.clear_output()


btn_compute.on_click(on_compute)
btn_save_mask.on_click(on_save_mask)
dd_rej.observe(_on_rej_select, names='value')
dd_psd_ch.observe(_on_psd_ch_change, names='value')
sig4.observe(reset_sec4, names='value')
on_load_callbacks.append(reset_sec4)

display(
    widgets.HBox([btn_compute, sig4]),
    HTML('<b>Peak-to-peak amplitude thresholds per stage</b>'),
    widgets.HBox([rj_W, rj_N1, rj_N2]), widgets.HBox([rj_N3, rj_R]),
    HTML('<b>Flat / gradient / 1f thresholds</b>'),
    widgets.HBox([rj_flat, rj_grad]), widgets.HBox([rj_1f_err, rj_1f_r2]),
    HTML('<b>Methods</b>'),
    widgets.HBox([cb_m_amp, cb_m_flat, cb_m_grad, cb_m_1f]),
    out_hm, out_summary,
    HTML('<hr><b>Mean PSD per stage — clean vs rejected</b>'),
    widgets.HBox([dd_psd_ch]), out_rejpsd,
    HTML('<hr><b>Rejected-epoch inspection</b>'),
    widgets.HBox([dd_rej]), out_rejview,
    HTML('<hr>'),
    widgets.HBox([btn_save_mask]), out_mask,
)